# Chapter 7 - Self-Evolution and Evaluation (worked example)

Companion notebook to `skills/self-evolution/`. This is the *cognitive autopsy* the
chapter builds toward: a DevOps agent makes a **partially correct** prediction, and the
self-evolution loop turns that failure into a shipped improvement without a human rewriting
the prompt by hand.

We thread the chapter's running scenario end to end (fictional AWS account `123456789012`,
all boto3 calls mocked with `moto`):

1. A routine dependency update (`stripe-python 3.2.1 -> 3.3.0`) triggers a **prediction**.
2. The **execution graph** captures the agent's reasoning as a queryable autobiography.
3. `moto`-mocked telemetry reveals the **actual** failure (connection pool exhaustion).
4. The **four-layer evaluation cascade** performs the cognitive autopsy.
5. We **classify** the evolution, **choose** the intervention, and **backpropagate** feedback.
6. The **graduated validation protocol** gates the fix to production; the entropy guard keeps the
   learning store from collapsing.
7. **XSkill / Cognee** turn the resolution into a self-improving skill object.

Every stage exercises a real skill from this repo (through its `lib.py` or its `cli.py`), and every
stage ends with an assert gate so the notebook fails loudly if the pipeline drifts.

## 1. Load the skills

Each skill ships its own `lib.py`. Because they are all named `lib`, importing them by bare name
would collide in `sys.modules`, so we load each one from its file path under a unique module name.
We also load the **existing** `execution-graph` skill: it produces the autobiography that every later
stage reads.

In [ ]:
import importlib.util
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SKILLS_DIR = PROJECT_ROOT / "skills" / "self-evolution"
FICTIONAL_ACCOUNT_ID = "123456789012"


def load_skill_lib(slug):
    lib_path = SKILLS_DIR / slug / "lib.py"
    mod_name = f"ch7_{slug.replace('-', '_')}"
    spec = importlib.util.spec_from_file_location(mod_name, lib_path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[mod_name] = mod
    spec.loader.exec_module(mod)
    return mod


exec_graph = load_skill_lib("execution-graph")
cascade = load_skill_lib("four-layer-eval-cascade")
taxonomy = load_skill_lib("evolution-taxonomy-classifier")
selector = load_skill_lib("intervention-selector")
backprop = load_skill_lib("semantic-backprop-attributor")
gvp = load_skill_lib("graduated-validation-protocol")
xskill = load_skill_lib("xskill-self-improving-object")

print("Loaded skills:")
for name in ("execution-graph", "four-layer-eval-cascade", "evolution-taxonomy-classifier",
             "intervention-selector", "semantic-backprop-attributor",
             "graduated-validation-protocol", "xskill-self-improving-object"):
    print(f"  - {name}")
print(f"\nFictional AWS account: {FICTIONAL_ACCOUNT_ID}")

## 2. The trigger: a dependency update and its blast radius

The platform team queues `stripe-python 3.2.1 -> 3.3.0` in `checkout-service`. Before the deploy goes
live, the agent computes the **blast radius** from the knowledge graph (chapter Example 7-14): which
services depend on the library, which methods they call, and what sits downstream.

We represent the Cypher result as a dict. The graph is the single source of truth the agent reasons
over; the whole point of Chapter 7 is that this structure is what makes later diagnosis possible.

In [ ]:
# Result of the KG dependency query (chapter Example 7-14).
blast_radius = {
    "service": "checkout-service",
    "library": "stripe-python",
    "version_change": "3.2.1 -> 3.3.0",
    "methods_used": ["payments.charge", "payments.retrieve"],
    "upstream": [],
    "downstream": ["order-service", "fulfillment-service"],
    "changelog_3_3_0": [
        "deprecated synchronous batch_charge API",
        "added async payment methods",
        "default connection timeout reduced from 30 to 10 seconds",
    ],
}
print("Blast radius for the stripe-python update:")
for k, v in blast_radius.items():
    print(f"  {k}: {v}")

# The agent's prediction reasons over this. It has BOTH signals in hand:
#   (a) a deprecated API (batch_charge)
#   (b) a configuration change (timeout 30s -> 10s)
# checkout-service does NOT call batch_charge, yet the agent will over-weight the API hypothesis.
assert "deprecated synchronous batch_charge API" in blast_radius["changelog_3_3_0"]
assert "default connection timeout reduced from 30 to 10 seconds" in blast_radius["changelog_3_3_0"]
print("\nGATE: both the API-deprecation and the timeout-config signals are present in the input.")

## 3. The actual outcome: `moto`-mocked telemetry

The SRE finds no `batch_charge` usage and approves the deploy. Twenty minutes later, connection
timeouts cascade through the order-processing pipeline. We reproduce that ground truth with `moto`:
real boto3 shapes, fictional account `123456789012`, zero credentials.

The seeded CloudWatch Logs events carry `db_pool_exhausted=true` after the deploy. This is the
**actual** failure mode (`connection pool exhaustion from the timeout reduction`) that the agent's
prediction misidentified. To target a real account, drop the `@mock_aws` decorator and the `AWS_*`
env vars; the boto3 code is unchanged.

In [ ]:
import os
import time

os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "testing")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "testing")

import boto3
from moto import mock_aws


@mock_aws
def observe_post_deploy_telemetry():
    logs = boto3.client("logs")
    log_group = "/aws/ecs/checkout-service"
    logs.create_log_group(logGroupName=log_group)
    logs.create_log_stream(logGroupName=log_group, logStreamName="task-3f3a9c")

    now_ms = int(time.time() * 1000)
    # Pre-deploy: healthy. Post-deploy (timeout 30s -> 10s): pool waits, then exhaustion.
    events = [
        {"timestamp": now_ms - 1_500_000, "message": "INFO checkout req=r001 duration_ms=41 status=200"},
        {"timestamp": now_ms - 600_000,   "message": "INFO deploy stripe-python 3.2.1->3.3.0 timeout=10s"},
        {"timestamp": now_ms - 300_000,   "message": "WARN checkout req=r107 duration_ms=2810 db_pool_wait_ms=2700"},
        {"timestamp": now_ms - 180_000,   "message": "ERROR checkout req=r113 duration_ms=5012 status=504 db_pool_exhausted=true"},
        {"timestamp": now_ms - 60_000,    "message": "ERROR checkout req=r119 duration_ms=4998 status=504 db_pool_exhausted=true"},
    ]
    logs.put_log_events(logGroupName=log_group, logStreamName="task-3f3a9c", logEvents=events)

    started = logs.start_query(
        logGroupName=log_group,
        startTime=int(now_ms / 1000) - 3600,
        endTime=int(now_ms / 1000),
        queryString=(
            "fields @timestamp, @message | filter db_pool_exhausted='true' "
            "or duration_ms > 1000 | sort @timestamp desc | limit 20"
        ),
    )
    exhausted = sum(1 for e in events if "db_pool_exhausted=true" in e["message"])
    return {"query_id": started["queryId"], "events_seeded": len(events),
            "pool_exhaustion_events": exhausted, "account": FICTIONAL_ACCOUNT_ID}

telemetry = observe_post_deploy_telemetry()
print("moto-mocked CloudWatch Logs telemetry:")
for k, v in telemetry.items():
    print(f"  {k}: {v}")

assert isinstance(telemetry["query_id"], str) and telemetry["query_id"]
assert telemetry["pool_exhaustion_events"] >= 2
print("\nGATE: telemetry confirms the ACTUAL failure is connection pool exhaustion, not an API break.")
ACTUAL_FAILURE = "Connection pool exhaustion from timeout reduction (30s->10s)"

## 4. The autobiography: build the execution graph

The prediction node's reasoning was captured by the OpenTelemetry instrumentation as an
**execution graph** (chapter's `execution-graph` skill). We rebuild the relevant sub-graph:

```
KnowledgeGraphQuery -> ChangelogRetrieval -> ChangeAnalyzer -> CausalAttributionNode -> PredictionSynthesis
```

The two-phase write is what lets us localize the failure to one node later. The
`CausalAttributionNode` is where the agent committed to the wrong hypothesis; its output is the
partially correct prediction `API contract violation`.

In [ ]:
g = exec_graph.ExecutionGraph(execution_id="pred_7f3a9c")
label_to_id = {}

def add(label, node_type, parent_label, output, latency_ms, tokens=0, cost=0.0):
    parent_id = label_to_id.get(parent_label)
    nid = g.begin_node(node_type, input_payload={"node": label}, parent_id=parent_id)
    g.complete_node(nid, output_payload=output, latency_ms=latency_ms,
                    token_count=tokens, cost_usd=cost)
    label_to_id[label] = nid
    return nid

add("KnowledgeGraphQuery", "Retrieval", None,
    {"deps": "checkout-service uses payments.charge/retrieve; no batch_charge usage"}, 98.7)
add("ChangelogRetrieval", "Retrieval", "KnowledgeGraphQuery",
    {"changelog": blast_radius["changelog_3_3_0"]}, 120.4)
add("ChangeAnalyzer", "LLM_Call", "ChangelogRetrieval",
    {"classified": "API deprecations present, configuration changes present"}, 610.2, 420, 0.006)
add("CausalAttributionNode", "LLM_Call", "ChangeAnalyzer",
    {"failure_mode": "API contract violation", "confidence": 0.78}, 1400.0, 847, 0.013)
add("PredictionSynthesis", "LLM_Call", "CausalAttributionNode",
    {"prediction": "HIGH impact; review checkout-service for batch_charge usage"}, 305.1, 210, 0.004)

summary = g.summary()
print("Execution graph summary:")
print(json.dumps(summary, indent=2, default=str))

chain = g.causal_chain(label_to_id["CausalAttributionNode"])
print("\nCausal chain to CausalAttributionNode:")
for node in chain:
    print(f"  <- {node.input_payload['node']} ({node.type})")

**Verification gate.** The graph must have all five prediction nodes and a causal chain from the
faulty node back to the root. Without the graph, the diagnosis in the next section would be guesswork.

In [ ]:
assert summary["node_count"] == 5, summary
chain_labels = [n.input_payload["node"] for n in chain]
assert chain_labels == ["KnowledgeGraphQuery", "ChangelogRetrieval", "ChangeAnalyzer", "CausalAttributionNode"]
assert summary["failed_node_count"] == 0  # nothing crashed; the failure is cognitive, not a crash
print("GATE: execution graph is complete and the causal chain to the faulty node resolves.")

## 5. The cognitive autopsy: four-layer evaluation cascade

Now the diagnostic engine runs. The cascade moves from the most general failure cause to the most
specific and **stops at the first failing layer**:

- **Layer 0** (hallucination gate): is the answer grounded in the retrieved premise?
- **Layer 1** (context evaluator): did the agent even possess the information it needed?
- **Layer 2** (cognitive fault isolator): knowledge failure or reasoning failure?
- **Layer 3** (TIR-judge): are the quantitative claims actually correct?

The agent had the right facts (Layer 1 passes, KI is high), so this is a **reasoning** failure. The
`InfoGain` trace `[0.34, 0.29, 0.22, 0.03, -0.01, 0.19]` shows the tell: steps 4 and 5 contribute
almost nothing. That is **premature closure**: the agent locked onto the API hypothesis and stopped
exploring the timeout hypothesis that had equal support.

In [ ]:
premise = (
    "Knowledge graph and changelog for stripe-python 3.3.0. "
    "Predicted analysis of the failure mode. "
    "Deprecated batch_charge API present. New async payment methods added. "
    "Default connection timeout reduced from 30 to 10 seconds. "
    "checkout-service uses payments charge and payments retrieve. "
    "API contract violation is one candidate. Connection pool exhaustion is another candidate."
)

execution = {
    "execution_id": "pred_7f3a9c",
    "query": "Will stripe-python 3.2.1 -> 3.3.0 cause failures in checkout-service?",
    "answer": "API contract violation from deprecated batch_charge",
    "context_premise": premise,
    "required_claims": ["deprecated batch_charge", "connection timeout reduced 30 to 10"],
    "infogain_trace": [0.34, 0.29, 0.22, 0.03, -0.01, 0.19],
    "knowledge_index": 0.91,
    "fault_node": {"node_id": "CausalAttributionNode", "node_type": "LLM_Call",
                   "description": "CausalAttributionNode"},
    "diagnosis": ("Premature commitment to 'API contract violation' at step 4. The configuration "
                  "change (timeout 30s -> 10s) had equal evidentiary support but was not explored "
                  "after the early commitment to the API path."),
}

report = cascade.run_cascade(execution)
print("Diagnostic report:")
print(json.dumps(report, indent=2, default=str))

**Verification gate.** The cascade must pass Layers 0 and 1 (the agent was grounded and had enough
context), stop at Layer 2 with a `REASONING` verdict, pinpoint the low-InfoGain steps `[4, 5]`, and
recommend the lightest fix - a prompt refinement.

In [ ]:
assert report["layer_1_context"]["sufficient"] is True
assert report["stopped_at_layer"] == 2
cog = report["layer_2_cognitive"]
assert cog["failure_type"] == "REASONING", cog
assert cog["low_infogain_steps"] == [4, 5], cog
assert abs(cog["knowledge_index"] - 0.91) < 1e-9
assert report["recommended_intervention"] == "PROMPT_REFINEMENT"
assert report["target_nodes"] == ["CausalAttributionNode"]
print("GATE: reasoning failure localized to CausalAttributionNode, steps 4-5, prompt refinement recommended.")

## 6. Classify the evolution across the four axes

Before pulling a lever, place the fix in the taxonomy (chapter's four-axis design space). Pulling the
wrong lever wastes compute or introduces regressions. Each axis value carries the graph-dependency
rationale the chapter gives for it.

In [ ]:
classification = taxonomy.classify_from_signals(
    what="model",              # we will change the prompt that drives reasoning
    when="inter_test_time",    # applied between requests, validated before the next run
    how="reward_based",        # driven by InfoGain / outcome signals
    where="domain_specialized",  # DevOps cascade prediction, not general capability
    notes="Prompt refinement for CausalAttributionNode after premature-closure diagnosis.",
)
print(json.dumps(classification.to_dict(), indent=2, default=str))

assert classification.what == "model" and classification.when == "inter_test_time"
assert classification.how == "reward_based" and classification.where == "domain_specialized"
assert set(classification.graph_rationale) == {"what", "when", "how", "where"}
print("\nGATE: evolution classified across all four axes with graph rationale attached.")

## 7. Choose the intervention (deterministic routing)

The diagnostic report already contains the failure type and target nodes, so intervention selection is
a deterministic, auditable routing function - not a judgment call each on-call engineer makes
differently. The same report the cascade produced feeds straight into the selector.

In [ ]:
intervention = selector.select_intervention(report)
print("Selected intervention:")
print(json.dumps(intervention.to_dict(), indent=2, default=str))
print("\nAudit line:")
print(" ", selector.explain(report))

print("\nIntensity hierarchy (why prompt refinement is the right first resort):")
for t in ("PROMPT_REFINEMENT", "FINE_TUNE", "CODE_MODIFICATION"):
    prof = selector.intervention_intensity(t)
    print(f"  {t:20s} risk={prof['risk']:8s} reversibility={prof['reversibility']}")

assert intervention.type == "PROMPT_REFINEMENT"
assert intervention.target == ["CausalAttributionNode"]
assert selector.risk_rank("PROMPT_REFINEMENT") < selector.risk_rank("FINE_TUNE") < selector.risk_rank("CODE_MODIFICATION")
print("\nGATE: intervention resolves to a prompt refinement scoped to the faulty node.")

## 8. Semantic backpropagation: neighbor-aware feedback

A single-node fix can ripple through neighbors. Semantic backpropagation prevents that by making the
feedback **neighbor-aware**: it includes the outputs of adjacent nodes so credit is assigned correctly.

First, the chapter's teaching example (Extractor -> CurrencyConverter -> Validator, with a DateChecker
feeding the Validator). The Validator surfaces the error, but attribution must land on the
`CurrencyConverter`'s rate lookup and leave the `Extractor` unchanged - because the DateChecker's
`Date: 2022` output is the neighbor evidence that exonerates the extraction.

In [ ]:
edges = [("Extractor", "CurrencyConverter"), ("CurrencyConverter", "Validator"),
         ("DateChecker", "Validator")]
node_outputs = {
    "Extractor": "Revenue: $10M",
    "CurrencyConverter": "EUR 9.5M",
    "DateChecker": "Date: 2022",
    "Validator": "conversion error flagged",
}
fb = backprop.backprop(
    edges, node_outputs, failure_node="Validator",
    predicted="EUR 9.5M",
    actual="EUR 9.0M; 2022 rate was 0.9 not 0.95",
)
print("Currency worked example - attributed feedback:")
print(json.dumps(fb.to_dict(), indent=2, default=str))

assert fb.target_node == "CurrencyConverter", "error must attribute to the converter, not the extractor"
assert "DateChecker_output" in fb.neighbor_context
assert "Extractor" not in fb.neighbor_context and fb.target_node != "Extractor"
print("\nGATE: neighbor-awareness attributes to CurrencyConverter and leaves the Extractor unchanged.")

Now the DevOps case, driven through the skill's **CLI** (`cli.py scenario devops-prediction`) to show
the same skill is harness-portable, not just importable. The feedback flowing backward to
`CausalAttributionNode` carries the neighbor evidence: the timeout change (30s -> 10s) *was* present in
the node's input, and no `batch_charge` usage was found - so the fix is a specific prompt change, not a
vague "consider more factors".

In [ ]:
import subprocess

cli = SKILLS_DIR / "semantic-backprop-attributor" / "cli.py"
proc = subprocess.run(
    [sys.executable, str(cli), "scenario", "devops-prediction"],
    cwd=str(PROJECT_ROOT), capture_output=True, text=True,
)
print(proc.stdout)
assert proc.returncode == 0, proc.stderr
out = proc.stdout.lower()
assert "causalattributionnode" in out
assert "30" in out and "10" in out  # the timeout neighbor evidence
print("GATE: CLI-driven semantic backprop emits neighbor-aware feedback for CausalAttributionNode.")

## 9. Graduated validation and the entropy-collapse guard

The candidate prompt refinement must not reach production unchecked. Because it is the lowest-risk
intervention category, it enters at **Tier 1 (canary)**: deploy to ~1% of traffic, require a
statistically significant lift with no core-KPI regression, auto-rollback otherwise. The provenance
signature records the change on the immutable ledger (RPO spine).

Separately, the **entropy-collapse guard** (Kepler dual-store) runs its daily garbage collection over
the ephemeral Learnings, removing anything promoted, contradicted, or stale past the 30-day TTL.

In [ ]:
candidate = {
    "intervention_type": "prompt_refinement",
    "target_node": "CausalAttributionNode",
    "metrics": {
        "target_lift": 0.12,          # +12% on premature-closure detection
        "target_pvalue": 0.01,        # statistically significant
        "kpi_deltas": {"latency": 0.0, "accuracy_other_predictions": 0.01},  # no regression
    },
}
result = gvp.graduated_validation(candidate)
print("Graduated validation:")
print(json.dumps(result.to_dict(), indent=2, default=str))
signature = gvp.provenance_signature(candidate)
print(f"\nProvenance signature (immutable ledger): {signature[:16]}...")

assert result.tier == "TIER1_CANARY"
assert result.passed is True
print("\nGATE: prompt refinement passes the Tier 1 canary with no KPI regression.")

# Entropy-collapse guard: daily GC over the Learnings subgraph.
Learning = gvp.Learning
learnings = [
    Learning("premature-closure-fix", confidence=0.9, last_accessed_days=1, resolved_promoted=True),
    Learning("stale-api-hypothesis", confidence=0.4, last_accessed_days=45),
    Learning("contradicted-guess", confidence=0.5, last_accessed_days=3, contradicted_by_higher_conf=True),
    Learning("fresh-pool-exhaustion-pattern", confidence=0.88, last_accessed_days=5),
]
kept, removed = gvp.garbage_collect(learnings)
print(f"\nGarbage collection: kept {[l.id for l in kept]}, removed {[l.id for l in removed]}")
rate = gvp.promotion_rate(total_learnings=len(learnings), promoted=1)
print(f"Promotion rate: {rate:.0%} -> {gvp.promotion_health(rate)}")

assert [l.id for l in kept] == ["fresh-pool-exhaustion-pattern"]
assert len(removed) == 3
print("GATE: GC keeps only the fresh, uncontradicted learning; the store does not collapse.")

## 10. Skills as self-improving graph objects (XSkill + Cognee)

The resolution pattern should not evaporate. **XSkill** distills action-level *experiences* from the
execution nodes, and **Cognee** treats skills as graph objects routed by **demonstrated success**, not
description similarity. After this incident a specialized `canary-deployment` skill with a strong track
record must outrank a general `deployment` skill that merely looks similar - even though the general
skill's description overlaps the task more.

In [ ]:
# XSkill: extract an action-level experience from the faulty node.
nodes = [
    {"id": "CausalAttributionNode", "task_type": "deployment_prediction",
     "action": "select primary failure mode", "context": "changelog has API + config changes",
     "outcome": "diverged", "caused_task_failure": True, "neighbors": ["ChangeAnalyzer"]},
    {"id": "PredictionSynthesis", "task_type": "deployment_prediction",
     "action": "emit structured prediction", "context": "single-hypothesis input",
     "outcome": "success", "caused_task_failure": False},
]
experiences = xskill.extract_experiences(nodes)
print("Extracted experiences (action-level):")
for e in experiences:
    print(f"  [{e.outcome}] {e.action_taken}: {e.lesson}")
assert len(experiences) == 1 and experiences[0].outcome == "failure"

# Cognee: route by demonstrated success, not description similarity.
graph = xskill.SkillGraph()
graph.add("deployment", {"name": "deployment",
                          "description": "general deployment skill for services",
                          "steps": ["check changelog", "deploy"]})
graph.add("canary-deployment", {"name": "canary deployment",
                                "description": "canary progressive rollout with pool checks",
                                "steps": ["canary 1 percent", "monitor kpis", "promote"]})

# Track record on the 'canary' pattern: general is weak (0.6), canary is strong (0.875).
for outcome in ["success", "success", "success", "failure", "failure"]:
    graph.learn("deployment", {"pattern": "canary", "description": "canary deployment"}, outcome)
for outcome in ["success"] * 7 + ["failure"]:
    graph.learn("canary-deployment", {"pattern": "canary", "description": "canary deployment"}, outcome)

task = {"description": "canary deployment for checkout-service", "pattern": "canary"}
chosen = graph.route(task)
print(f"\nRouting a canary task:")
print(f"  deployment        success@canary = {graph.skills['deployment'].success_rate_for('canary'):.3f}")
print(f"  canary-deployment success@canary = {graph.skills['canary-deployment'].success_rate_for('canary'):.3f}")
print(f"  -> route() picked: {chosen.skill_id}")

assert chosen.skill_id == "canary-deployment", "must route by demonstrated success, not description"
print("\nGATE: routing selects the skill with the better track record, not the closer description.")

## 11. Closing verdict: the loop closed

This is the complete self-evolution loop from the chapter, run end to end:

`execution -> diagnosis -> classification -> intervention -> feedback -> validation -> deployment`

A partially correct prediction that would previously have required a human to analyze the reasoning
gap and rewrite the prompt instead triggered an automated loop. The execution graph captured what
happened, the four-layer cascade located the premature closure, semantic backpropagation generated
precise neighbor-aware feedback, and a prompt refinement passed the Tier 1 canary. The SRE team
reviews the change at canary validation, freed from figuring out *what* to fix and asked only to
approve *that* it should be fixed.

In [ ]:
shipped_improvement = {
    "target_node": report["target_nodes"][0],
    "intervention": intervention.type,
    "prompt_change": ("When a changelog contains both API changes and configuration changes, "
                      "evaluate each as a separate hypothesis and assign independent confidence "
                      "scores before selecting a primary failure mode."),
    "diagnosis": report["layer_2_cognitive"]["diagnosis"],
    "actual_failure": ACTUAL_FAILURE,
}
validation_record = {
    "tier": result.tier,
    "passed": result.passed,
    "reasons": result.reasons,
    "provenance_signature": signature[:16] + "...",
    "evolution_axes": classification.to_dict(),
}

print("SHIPPED IMPROVEMENT")
print(json.dumps(shipped_improvement, indent=2, default=str))
print("\nVALIDATION RECORD")
print(json.dumps(validation_record, indent=2, default=str))

# Final end-to-end gate: every stage of the loop produced its expected artifact.
assert report["recommended_intervention"] == "PROMPT_REFINEMENT"
assert intervention.type == "PROMPT_REFINEMENT"
assert classification.what == "model"
assert result.tier == "TIER1_CANARY" and result.passed
assert chosen.skill_id == "canary-deployment"
assert telemetry["pool_exhaustion_events"] >= 2
print("\n" + "=" * 68)
print("ALL CH7 SELF-EVOLUTION GATES PASSED - the loop closed end to end.")
print("=" * 68)